In [105]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
import sqlite3
import os

#SCRAPING

Collected 100 books here I have followed an approach where I collected all the books in the first 5 pages.

In [57]:
all_books=[]
for i in range(1,6):
  if i==1:
    url = "https://books.toscrape.com/"
  else:
    url = f"https://books.toscrape.com/catalogue/page-{i}.html"
  response = requests.get(url)
  response.raise_for_status()
  soup = BeautifulSoup(response.text, 'html.parser')
  books= soup.find_all('article', class_= 'product_pod')
  for book in books:
    book_title = book.select_one("h3 a").get_text()
    price =  book.select_one("p.price_color").get_text(strip=True)
    rating = book.find('p',class_='star-rating')['class'][1]
    availability = book.select_one("p.instock").get_text(strip=True)
    book_link = book.select_one("h3 a")["href"]
    book_url=urljoin(url, book_link)
    response_book = requests.get(book_url)
    response_book.raise_for_status()
    soup_book = BeautifulSoup(response_book.text, 'html.parser')
    category=soup_book.select("ul.breadcrumb li")[-2].select_one("a").get_text(strip=True)
    all_books.append({
      "title":book_title,
      "price":price,
      "star_rating":rating,
      "availability":availability,
      "category":category
    })






In [58]:
print(len(all_books))

100


#CLEANING

In [107]:
df=pd.DataFrame(all_books)

mapping={"One":"1","Two":"2","Three":"3","Four":"4","Five":"5"}
df["price_gbp"]=df["price"].replace({"Â£":""},regex=True).astype(float)
df.drop(columns=["price"],inplace=True)
df['star_rating']=df['star_rating'].map(mapping).astype(int)
df.rename(columns={"star_rating":"rating"},inplace=True)
df["availability"]=df["availability"]=="In stock"
df.rename(columns={"availability":"in_stock"},inplace=True)


df["price_gbp"]=pd.to_numeric(df["price_gbp"],errors="coerce")
missing_percentage = (df["price_gbp"].isna().sum()/ len(df["price_gbp"])) * 100
if missing_percentage < 5:
    df.dropna(subset=["price_gbp"], inplace=True)
else:
    df["price_gbp"] = df["price_gbp"].fillna(df["price_gbp"].median())

df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
missing_percentage = (df["rating"].isna().sum()/ len(df["rating"])) * 100
if missing_percentage < 5:
    df.dropna(subset=["rating"], inplace=True)
else:
    df["rating"] = df["rating"].fillna(df["rating"].median()).round().astype(int)


df=df.dropna(subset=["category","title"])

df["price_inr"]=(df["price_gbp"]*105.50).round(2)
df


,title,rating,in_stock,category,price_gbp,price_inr
0,A Light in the ...,3,True,Poetry,51.77,5461.74
1,Tipping the Velvet,1,True,Historical Fiction,53.74,5669.57
2,Soumission,1,True,Fiction,50.10,5285.55
3,Sharp Objects,4,True,Mystery,47.82,5045.01
4,Sapiens: A Brief History ...,5,True,History,54.23,5721.26
...,...,...,...,...,...,...
95,Lumberjanes Vol. 3: A ...,2,True,Sequential Art,19.92,2101.56
96,"Layered: Baking, Building, and ...",1,True,Food and Drink,40.11,4231.60
97,Judo: Seven Steps to ...,2,True,Add a comment,53.90,5686.45
98,Join,5,True,Science Fiction,35.67,3763.19


For categorical columns like category and title if there is any missing field I directly dropped the row

For numerical columns like price and rating.
1) First converted Invalid numeric values to missing values (`NaN`) using
`pd.to_numeric(..., errors="coerce")`
2) Calculated the percentage of missing. If the percentage of data missing less than 5%, then I dropped the rows else did median imputation.

Currency Conversion normal fixed way no external API. For inr I did round for 2 decimal points


In [87]:
print(df.isna().sum())

title           0
rating          0
availability    0
category        0
price_gbp       0
price_inr       0
dtype: int64


#DATABASE

In [138]:
db_path = "zepto_books.db"

if os.path.exists(db_path):
  os.remove(db_path)

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT UNIQUE NOT NULL
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    price_gbp REAL,
    price_inr REAL,
    rating INTEGER,
    in_stock INTEGER,
    category_id INTEGER,
    FOREIGN KEY (category_id) REFERENCES categories(category_id)
)
""")
conn.commit()



In [115]:
categories=df["category"].unique()

for category in categories:
  cursor.execute("INSERT OR IGNORE INTO categories (category_name) VALUES (?)",(category,))
conn.commit()

In [114]:
categories_df= pd.read_sql("SELECT * FROM categories",conn)


,category_id,category_name
0,1,Poetry
1,2,Historical Fiction
2,3,Fiction
3,4,Mystery
4,5,History
5,6,Young Adult
6,7,Business
7,8,Default
8,9,Sequential Art
9,10,Music


In [116]:
mapping_category=dict(zip(categories_df["category_name"],categories_df["category_id"]))
df["category_id"]=df["category"].map(mapping_category)

In [127]:
for _,row in df.iterrows():
  cursor.execute("""
    INSERT INTO books (
        title,
        price_gbp,
        price_inr,
        rating,
        in_stock,
        category_id
    )
    VALUES (?, ?, ?, ?, ?, ?)
""", (row["title"],row["price_gbp"],row["price_inr"],int(row["rating"]),int(row["in_stock"]),int(row["category_id"])))

conn.commit()

In [128]:
query1 = pd.read_sql("""
SELECT title, price_gbp
FROM books
WHERE price_gbp > 40;
""",conn)

query2 = pd.read_sql("""
SELECT title, rating
FROM books
ORDER BY rating DESC;
""",conn)

query3 = pd.read_sql("""
SELECT title, price_gbp
FROM books
ORDER BY price_gbp DESC
LIMIT 10;
""",conn)

query4 = pd.read_sql("""
SELECT DISTINCT rating
FROM books;
""",conn)

query5 = pd.read_sql("""
SELECT title, price_gbp
FROM books
WHERE price_gbp BETWEEN 20 AND 40;
""",conn)

join_query = pd.read_sql("""
SELECT b.title,b.price_gbp,b.rating,c.category_name
FROM books b
JOIN categories c
ON b.category_id = c.category_id
ORDER BY b.rating DESC, b.price_gbp DESC
LIMIT 10;
""",conn)

In [135]:
print(query1)

                                   title  price_gbp
0                     A Light in the ...      51.77
1                     A Light in the ...      51.77
2                     Tipping the Velvet      53.74
3                             Soumission      50.10
4                          Sharp Objects      47.82
..                                   ...        ...
116                    Masks and Shadows      56.40
117  Lumberjanes, Vol. 2: Friendship ...      46.91
118      Lumberjanes, Vol. 1: Beware ...      45.61
119   Layered: Baking, Building, and ...      40.11
120             Judo: Seven Steps to ...      53.90

[121 rows x 2 columns]


In [136]:
print(query2)

                                   title  rating
0           Sapiens: A Brief History ...       5
1                            Set Me Free       5
2    Scott Pilgrim's Precious Little ...       5
3                      Rip it Up and ...       5
4             Chase Me (Paris Nights ...       5
..                                   ...     ...
296               The Age of Genius: ...       1
297              Pop Gun War, Volume ...       1
298  orange: The Complete Collection ...       1
299        Online Marketing for Busy ...       1
300   Layered: Baking, Building, and ...       1

[301 rows x 2 columns]


In [137]:
print(query3)

                          title  price_gbp
0    The Death of Humanity: ...      58.11
1    The Death of Humanity: ...      58.11
2    The Death of Humanity: ...      58.11
3  Slow States of Collapse: ...      57.31
4  Slow States of Collapse: ...      57.31
5  Slow States of Collapse: ...      57.31
6         Our Band Could Be ...      57.25
7         Our Band Could Be ...      57.25
8         Our Band Could Be ...      57.25
9           The Past Never Ends      56.50


In [130]:
print(join_query)

                                 title  price_gbp  rating   category_name
0         Sapiens: A Brief History ...      54.23       5         History
1         Sapiens: A Brief History ...      54.23       5         History
2         Sapiens: A Brief History ...      54.23       5         History
3  Scott Pilgrim's Precious Little ...      52.29       5  Sequential Art
4  Scott Pilgrim's Precious Little ...      52.29       5  Sequential Art
5  Scott Pilgrim's Precious Little ...      52.29       5  Sequential Art
6             We Love You, Charlie ...      50.27       5         Fiction
7             We Love You, Charlie ...      50.27       5         Fiction
8             We Love You, Charlie ...      50.27       5         Fiction
9          Private Paris (Private #10)      47.61       5         Fiction


In [134]:
books_df = pd.read_sql("SELECT * FROM books",conn)
categories_df = pd.read_sql("SELECT * FROM categories",conn)
join_df= pd.merge(books_df,categories_df, on="category_id",how="inner")
final_joined_df=join_df[["title", "price_gbp", "rating", "category_name"]].sort_values(by=["rating","price_gbp"], ascending=[False, False]).reset_index(drop=True).head(10)
print(final_joined_df)


                                 title  price_gbp  rating   category_name
0         Sapiens: A Brief History ...      54.23       5         History
1         Sapiens: A Brief History ...      54.23       5         History
2         Sapiens: A Brief History ...      54.23       5         History
3  Scott Pilgrim's Precious Little ...      52.29       5  Sequential Art
4  Scott Pilgrim's Precious Little ...      52.29       5  Sequential Art
5  Scott Pilgrim's Precious Little ...      52.29       5  Sequential Art
6             We Love You, Charlie ...      50.27       5         Fiction
7             We Love You, Charlie ...      50.27       5         Fiction
8             We Love You, Charlie ...      50.27       5         Fiction
9          Private Paris (Private #10)      47.61       5         Fiction
